# 02｜資料準備：清理、探索與轉換

本 notebook 讀取專案中的 `fccl_thickness_dataset.csv`，執行資料品質檢查、探索性資料分析（EDA）、特徵轉換，最後匯出 CSV。資料為合成 FCCL 製程紀錄，刻意注入的異常、QC 標記及原有缺值都會保留，不會無依據地刪除或填補。

輸出位置：`data/processed/`。請依序執行所有程式碼儲存格。

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager

# 從目前目錄或上層目錄尋找原始資料。
source_name = "fccl_thickness_dataset.csv"
source_path = next((p / source_name for p in [Path.cwd(), *Path.cwd().parents] if (p / source_name).exists()), None)
if source_path is None:
    raise FileNotFoundError(f"找不到 {source_name}，請確認目前工作目錄位於專案內。")
output_dir = source_path.parent / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# 優先使用系統已安裝的中文字型，確保圖表標題和座標軸能顯示中文。
font_candidates = ["Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC", "Noto Sans CJK SC", "PingFang TC", "SimHei", "Arial Unicode MS"]
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next((name for name in font_candidates if name in installed_fonts), None)
if chinese_font:
    plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
else:
    warnings.warn("找不到中文字型，圖表中文可能顯示為方框。請安裝 Microsoft JhengHei 或 Noto Sans CJK TC，重新啟動 kernel 後再執行。")
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")
print(f"來源檔案：{source_path}")
print(f"輸出資料夾：{output_dir}")
print(f"圖表中文字型：{chinese_font or '未找到'}")

## 1. 載入與初步檢視

查看資料規模、欄位型態、缺值與重複紀錄，並將樣本日期轉為日期型態。

In [ ]:
df = pd.read_csv(source_path)
print(f"資料列數：{len(df):,}；欄位數：{df.shape[1]}")
display(df.head())
display(pd.DataFrame({"資料型態": df.dtypes.astype(str), "缺值數": df.isna().sum(), "缺值比例": df.isna().mean().round(3)}))
print(f"完全重複列：{df.duplicated().sum()}")
if "Batch_ID" in df:
    print(f"重複批次 ID：{df['Batch_ID'].duplicated().sum()}")
df["Sample_Date"] = pd.to_datetime(df["Sample_Date"], errors="coerce")
print(f"無法解析的樣本日期：{df['Sample_Date'].isna().sum()}")

## 2. 資料清理與品質檢查

修剪文字前後空白、移除完全重複列，保留原始缺值和異常標記。檢查厚度摘要是否與 10 個量測點一致，以及規格界限是否對稱。

In [ ]:
text_cols = df.select_dtypes(include=["object", "str"]).columns
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())
df = df.drop_duplicates().copy()
point_cols = [f"Point_{i}_um" for i in range(1, 11)]
points = df[point_cols]
checks = pd.DataFrame({
    "平均值相符（誤差 ≤ 0.001）": (points.mean(axis=1) - df["Thickness_Mean_um"]).abs() <= 0.001,
    "標準差相符（誤差 ≤ 0.001）": (points.std(axis=1) - df["Thickness_Std_um"]).abs() <= 0.001,
    "全距相符（誤差 ≤ 0.001）": (points.max(axis=1) - points.min(axis=1) - df["Thickness_Range_um"]).abs() <= 0.001,
    "規格界限對稱": ((df["Nominal_Thickness_um"] - df["LSL_um"]) - (df["USL_um"] - df["Nominal_Thickness_um"])).abs() <= 0.001,
})
display(pd.DataFrame({"通過筆數": checks.sum(), "未通過筆數": (~checks).sum()}))
display(df.isna().sum().loc[lambda s: s.gt(0)].rename("保留缺值數").to_frame())
display(df["Injected_Anomaly"].value_counts(dropna=False).rename("筆數").to_frame())
display(df["QC_Result"].value_counts(dropna=False).rename("筆數").to_frame())

## 3. 探索性資料分析（EDA）

下方以中文標示批次平均厚度分布、每月注入異常比例和各產線 QC 結果比例。圖表中文字型已在第一個儲存格設定。

In [ ]:
display(pd.Series({"批次數": len(df), "產品型號數": df["Product_Model"].nunique(), "產線數": df["Line"].nunique(), "注入異常批次數": int(df["Injected_Anomaly"].sum()), "NG 批次數": int(df["QC_Result"].eq("NG").sum())}, name="數值").to_frame())
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.histplot(df["Thickness_Mean_um"], bins=30, kde=True, ax=axes[0], color="#4472C4")
axes[0].set(title="批次平均厚度分布", xlabel="平均厚度（µm）", ylabel="批次數")
monthly = df.groupby("Month", sort=True)["Injected_Anomaly"].mean().mul(100).rename("異常比例").reset_index()
sns.barplot(data=monthly, x="Month", y="異常比例", ax=axes[1], color="#ED7D31")
axes[1].tick_params(axis="x", rotation=60)
axes[1].set(title="每月注入異常比例", xlabel="月份", ylabel="異常比例（%）")
qc = pd.crosstab(df["Line"], df["QC_Result"], normalize="index").mul(100)
colors = [{"NG": "#C00000", "PASS": "#70AD47"}.get(label, "#4472C4") for label in qc.columns]
qc.plot(kind="bar", stacked=True, ax=axes[2], color=colors)
axes[2].set(title="各產線 QC 結果比例", xlabel="產線", ylabel="批次比例（%）")
axes[2].legend(title="QC 結果", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
display(pd.crosstab(df["Anomaly_Pattern"], df["QC_Result"], margins=True).rename_axis("異常模式"))

## 4. 特徵轉換

產生批次層級清理資料、每個量測點一列的長格式資料，以及實際值相對目標值的偏差和偏差百分比。偏差定義為「實際值 − 目標值」，缺值維持空白。

In [ ]:
# 清理資料：格式化日期，保留其他來源欄位。
cleaned = df.copy()
cleaned["Sample_Date"] = cleaned["Sample_Date"].dt.strftime("%Y-%m-%d")

# 將 10 個測點攤平成長格式，並依批次規格判定是否合格。
measurements = df.melt(id_vars=[c for c in df.columns if c not in point_cols], value_vars=point_cols, var_name="Measurement_Point", value_name="Measured_Thickness_um")
measurements["Measurement_Point_Number"] = measurements["Measurement_Point"].str.extract(r"(\d+)").astype(int)
measurements["Within_Spec"] = measurements["Measured_Thickness_um"].between(measurements["LSL_um"], measurements["USL_um"], inclusive="both")
measurements["Sample_Date"] = measurements["Sample_Date"].dt.strftime("%Y-%m-%d")

# 對每組目標值和實際值計算差值與百分比差值。
process_features = df[["Batch_ID", "Sample_Date", "Month", "Line", "Product_Model", "Injected_Anomaly", "QC_Result"]].copy()
for target_col in [c for c in df.columns if c.endswith("_target")]:
    base = target_col.removesuffix("_target")
    actual_col = f"{base}_actual"
    if actual_col in df.columns:
        process_features[f"{base}_deviation"] = df[actual_col] - df[target_col]
        process_features[f"{base}_deviation_pct"] = np.where(df[target_col].ne(0), (df[actual_col] - df[target_col]) / df[target_col] * 100, np.nan)
process_features["Sample_Date"] = process_features["Sample_Date"].dt.strftime("%Y-%m-%d")
display(measurements[["Batch_ID", "Measurement_Point", "Measured_Thickness_um", "Within_Spec"]].head())
display(process_features.head())

## 5. 匯出 CSV

使用 UTF-8 with BOM（`utf-8-sig`）編碼，方便 Windows Excel 正確辨識文字。重複執行會覆寫同名檔案。

In [ ]:
outputs = {"fccl_cleaned.csv": cleaned, "fccl_measurements_long.csv": measurements, "fccl_process_deviation_features.csv": process_features}
for filename, data in outputs.items():
    destination = output_dir / filename
    data.to_csv(destination, index=False, encoding="utf-8-sig")
    print(f"已匯出：{destination}（{len(data):,} 列 × {data.shape[1]} 欄）")

## 輸出檔案與注意事項

- `fccl_cleaned.csv`：保留原始欄位的批次層級清理資料。
- `fccl_measurements_long.csv`：厚度測點長格式資料，每批最多 10 列。
- `fccl_process_deviation_features.csv`：批次層級製程偏差特徵。
- 溫度缺值、異常標記與 QC 結果均會保留。`Potential_Root_Cause_Synthetic` 是合成資料中的假設，不代表實際調查結論。
- 若中文仍顯示為方框，請安裝 Microsoft JhengHei 或 Noto Sans CJK TC，重新啟動 Jupyter kernel 後再執行。